
# Amazon Search — Week 7 V3
## Streamlit Search Demo — One Click

This notebook builds the demo UI on top of the **validated Week 7 V2 FastAPI package**.

### V2 baseline already passed
- 10,000-product catalog
- FAISS Top-100 retrieval
- 29-feature saved LambdaMART
- API mean round-trip latency: **~92.6 ms**
- API P95 latency: **~119.6 ms**
- All V2 validation checks passed

### Upload only one file
`amazon_week7_v2_fastapi_service.zip`

### V3 demo UI
A user can:
1. type a natural-language query;
2. choose Top-K results;
3. receive ranked products;
4. view title, brand/category, price, rating, image when available;
5. view LambdaMART score and SBERT retrieval score;
6. inspect stage-level latency.

The generated package supports:
- **Direct mode** — Streamlit loads `SearchEngine` directly (easiest local demo);
- **API mode** — Streamlit calls the FastAPI `/search` endpoint via `API_URL`.


In [ ]:

# 1. Install dependencies
!pip -q install \
    streamlit fastapi uvicorn httpx requests \
    pandas pyarrow numpy scikit-learn faiss-cpu lightgbm \
    sentence-transformers rank-bm25 transformers accelerate

print("✅ Dependencies installed")


In [ ]:

# 2. Upload the validated Week 7 V2 service ZIP
from google.colab import files
from pathlib import Path
import zipfile, shutil, os, json, re, time
import pandas as pd
import numpy as np

ROOT = Path("/content/amazon_week7_v3")
if ROOT.exists():
    shutil.rmtree(ROOT)

ROOT.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

zip_candidates = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]

v2_zip = None
for name in zip_candidates:
    low = name.lower()
    if "week7" in low and "v2" in low and "fastapi" in low:
        v2_zip = name
        break

if v2_zip is None:
    if len(zip_candidates) == 1:
        v2_zip = zip_candidates[0]
    else:
        raise ValueError(
            "Please upload amazon_week7_v2_fastapi_service.zip"
        )

zip_path = ROOT / v2_zip
zip_path.write_bytes(uploaded[v2_zip])

SERVICE_ROOT = ROOT / "amazon_week7_v3_demo"
SERVICE_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(SERVICE_ROOT)

print("✅ Extracted:", v2_zip)
print("\nPackage root:")
for p in sorted(SERVICE_ROOT.iterdir()):
    print(" -", p.name)


In [ ]:

# 3. Verify V2 package before adding the UI
required_paths = [
    SERVICE_ROOT / "amazon_search_service" / "search_engine.py",
    SERVICE_ROOT / "amazon_search_service" / "main.py",
    SERVICE_ROOT / "artifacts" / "catalog.parquet",
    SERVICE_ROOT / "artifacts" / "sbert_product_embeddings.npy",
    SERVICE_ROOT / "artifacts" / "clip_image_embeddings.npy",
    SERVICE_ROOT / "artifacts" / "clip_embedding_map.csv",
    SERVICE_ROOT / "artifacts" / "lambdamart_model.txt",
    SERVICE_ROOT / "artifacts" / "runtime_config.json",
]

missing = [str(p.relative_to(SERVICE_ROOT)) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "V2 ZIP is missing required files:\n" + "\n".join(missing)
    )

with open(SERVICE_ROOT / "artifacts" / "runtime_config.json") as f:
    runtime_cfg = json.load(f)

catalog = pd.read_parquet(SERVICE_ROOT / "artifacts" / "catalog.parquet")

print("✅ V2 package verified")
print(f"✅ Catalog products: {len(catalog):,}")
print(f"✅ Ranker features: {len(runtime_cfg['feature_cols'])}")


In [ ]:

# 4. Create Streamlit application
STREAMLIT_SOURCE = r"""
import os
import time
from pathlib import Path

import pandas as pd
import requests
import streamlit as st

st.set_page_config(
    page_title="Amazon Search Ranking Demo",
    page_icon="🔎",
    layout="wide",
)

ROOT = Path(__file__).resolve().parent
API_URL = os.getenv("API_URL", "").strip().rstrip("/")


@st.cache_resource(show_spinner="Loading search models...")
def load_engine():
    from amazon_search_service.search_engine import SearchEngine
    return SearchEngine(
        artifact_dir=ROOT / "artifacts"
    )


def direct_search(query, top_k, candidate_k):
    engine = load_engine()
    return engine.search(
        query=query,
        top_k=top_k,
        candidate_k=candidate_k,
    )


def api_search(query, top_k, candidate_k):
    response = requests.post(
        f"{API_URL}/search",
        json={
            "query": query,
            "top_k": top_k,
            "candidate_k": candidate_k,
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def do_search(query, top_k, candidate_k):
    if API_URL:
        return api_search(
            query,
            top_k,
            candidate_k
        )
    return direct_search(
        query,
        top_k,
        candidate_k
    )


def money(value):
    if value is None:
        return "N/A"
    try:
        return f"${float(value):,.2f}"
    except Exception:
        return "N/A"


def rating_text(value):
    if value is None:
        return "N/A"
    try:
        return f"{float(value):.1f} / 5"
    except Exception:
        return "N/A"


st.title("Amazon Search Ranking Demo")
st.caption(
    "SBERT + FAISS Top-100 retrieval → "
    "29-feature LambdaMART re-ranking"
)

backend_name = (
    "FastAPI"
    if API_URL
    else "Direct SearchEngine"
)

with st.sidebar:
    st.header("Search settings")

    top_k = st.slider(
        "Results to return",
        min_value=3,
        max_value=20,
        value=10,
        step=1,
    )

    candidate_k = st.select_slider(
        "Retrieval candidates",
        options=[50, 75, 100, 150, 200],
        value=100,
    )

    st.divider()

    st.write("**Backend**")
    st.write(backend_name)

    st.write("**Catalog**")
    st.write("10,000 products")

    st.write("**Ranker**")
    st.write("LambdaMART — 29 features")

    st.write("**Week 7 target**")
    st.write("< 200 ms")

query = st.text_input(
    "Search products",
    value="best non drying lip stain",
    placeholder="Type a natural-language product query...",
)

search_clicked = st.button(
    "Search",
    type="primary",
    use_container_width=True,
)

if search_clicked:
    if not query.strip():
        st.warning("Please enter a search query.")
        st.stop()

    with st.spinner("Retrieving and re-ranking products..."):
        ui_start = time.perf_counter()

        try:
            payload = do_search(
                query.strip(),
                top_k,
                candidate_k,
            )
        except Exception as e:
            st.error(
                f"Search failed: {type(e).__name__}: {e}"
            )
            st.stop()

        ui_total_ms = (
            time.perf_counter() - ui_start
        ) * 1000

    latency = payload.get(
        "latency_ms",
        {}
    )

    total_ms = float(
        latency.get("total", ui_total_ms)
    )

    if total_ms < 200:
        latency_status = "PASS"
    else:
        latency_status = "ABOVE TARGET"

    c1, c2, c3, c4 = st.columns(4)

    c1.metric(
        "Results",
        len(payload.get("results", []))
    )

    c2.metric(
        "Pipeline latency",
        f"{total_ms:.1f} ms"
    )

    c3.metric(
        "UI round trip",
        f"{ui_total_ms:.1f} ms"
    )

    c4.metric(
        "<200 ms target",
        latency_status
    )

    with st.expander(
        "Latency breakdown",
        expanded=False
    ):
        latency_rows = []

        for name in [
            "embedding",
            "retrieval",
            "features",
            "ranking",
            "total",
        ]:
            if name in latency:
                latency_rows.append({
                    "stage": name,
                    "latency_ms": round(
                        float(latency[name]),
                        3
                    )
                })

        if latency_rows:
            st.dataframe(
                pd.DataFrame(latency_rows),
                hide_index=True,
                use_container_width=True,
            )

    st.subheader(
        f"Ranked results for “{payload.get('query', query)}”"
    )

    results = payload.get(
        "results",
        []
    )

    if not results:
        st.info("No products returned.")
        st.stop()

    for item in results:
        rank = item.get("rank", "")
        title = item.get(
            "title",
            "Untitled product"
        )

        with st.container(border=True):
            image_col, text_col = st.columns(
                [1, 4]
            )

            image_url = item.get(
                "image_url"
            )

            with image_col:
                if image_url:
                    try:
                        st.image(
                            image_url,
                            use_container_width=True
                        )
                    except Exception:
                        st.write("No image")
                else:
                    st.write("No image")

            with text_col:
                st.markdown(
                    f"### #{rank} {title}"
                )

                brand = (
                    item.get("brand")
                    or "Unknown brand"
                )

                category = (
                    item.get("category")
                    or "Unknown category"
                )

                st.caption(
                    f"{brand} · {category}"
                )

                m1, m2, m3, m4 = st.columns(4)

                m1.metric(
                    "Price",
                    money(item.get("price"))
                )

                m2.metric(
                    "Rating",
                    rating_text(
                        item.get("average_rating")
                    )
                )

                reviews = item.get(
                    "review_count"
                )

                if reviews is None:
                    review_text = "N/A"
                else:
                    try:
                        review_text = f"{int(float(reviews)):,}"
                    except Exception:
                        review_text = "N/A"

                m3.metric(
                    "Reviews",
                    review_text
                )

                m4.metric(
                    "Rank score",
                    f"{float(item.get('ranking_score', 0)):.4f}"
                )

                st.caption(
                    "SBERT retrieval score: "
                    f"{float(item.get('retrieval_score', 0)):.4f}"
                )

    with st.expander(
        "Raw JSON-compatible result table",
        expanded=False
    ):
        st.dataframe(
            pd.DataFrame(results),
            hide_index=True,
            use_container_width=True,
        )

else:
    st.info(
        "Enter a natural-language query and click Search."
    )
"""

(SERVICE_ROOT / "streamlit_app.py").write_text(
    STREAMLIT_SOURCE,
    encoding="utf-8"
)

print("✅ streamlit_app.py created")


In [ ]:

# 5. Add launch scripts and Docker Compose
RUN_DEMO_SOURCE = r"""
import os
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parent

api = subprocess.Popen([
    sys.executable,
    "-m",
    "uvicorn",
    "amazon_search_service.main:app",
    "--host",
    "127.0.0.1",
    "--port",
    "8000",
], cwd=ROOT)

time.sleep(3)

env = os.environ.copy()
env["API_URL"] = "http://127.0.0.1:8000"

try:
    ui = subprocess.Popen([
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "streamlit_app.py",
        "--server.address",
        "0.0.0.0",
        "--server.port",
        "8501",
    ], cwd=ROOT, env=env)

    ui.wait()

finally:
    api.terminate()
"""

DOCKER_COMPOSE = r"""
services:
  api:
    build: .
    command:
      - uvicorn
      - amazon_search_service.main:app
      - --host
      - 0.0.0.0
      - --port
      - "8000"
    ports:
      - "8000:8000"

  ui:
    build: .
    command:
      - streamlit
      - run
      - streamlit_app.py
      - --server.address
      - 0.0.0.0
      - --server.port
      - "8501"
    environment:
      API_URL: http://api:8000
    ports:
      - "8501:8501"
    depends_on:
      - api
"""

requirements_path = SERVICE_ROOT / "requirements.txt"
existing = requirements_path.read_text(encoding="utf-8")

extra = []
for package in ["streamlit", "requests"]:
    if package not in existing.lower():
        extra.append(package)

if extra:
    requirements_path.write_text(
        existing.rstrip()
        + "\n"
        + "\n".join(extra)
        + "\n",
        encoding="utf-8"
    )

(SERVICE_ROOT / "run_demo.py").write_text(
    RUN_DEMO_SOURCE,
    encoding="utf-8"
)

(SERVICE_ROOT / "docker-compose.yml").write_text(
    DOCKER_COMPOSE.strip() + "\n",
    encoding="utf-8"
)

print("✅ run_demo.py created")
print("✅ docker-compose.yml created")
print("✅ requirements.txt updated")


In [ ]:

# 6. Add Streamlit documentation
readme_path = SERVICE_ROOT / "README.md"
existing_readme = (
    readme_path.read_text(encoding="utf-8")
    if readme_path.exists()
    else ""
)

v3_docs = r"""

---

# Week 7 V3 — Streamlit Demo

## Direct UI mode

The simplest demo path loads the optimized SearchEngine directly:

```bash
streamlit run streamlit_app.py
```

Open:

```text
http://localhost:8501
```

## API-backed UI mode

Terminal 1:

```bash
uvicorn amazon_search_service.main:app --host 127.0.0.1 --port 8000
```

Terminal 2:

### macOS / Linux

```bash
API_URL=http://127.0.0.1:8000 streamlit run streamlit_app.py
```

### Windows PowerShell

```powershell
$env:API_URL="http://127.0.0.1:8000"
streamlit run streamlit_app.py
```

## One-command local launcher

```bash
python run_demo.py
```

This starts:
- FastAPI: `http://127.0.0.1:8000`
- Streamlit: `http://localhost:8501`

## Docker Compose

```bash
docker compose up --build
```

Then open:
- Streamlit: `http://localhost:8501`
- FastAPI docs: `http://localhost:8000/docs`

## UI contents

The Streamlit interface displays:

- natural-language query input;
- Top-K and candidate count controls;
- ranked product title;
- brand and category;
- price, rating, review count;
- product image when the catalog contains an image URL;
- LambdaMART ranking score;
- SBERT retrieval score;
- stage-level latency;
- `<200 ms` target status.
"""

readme_path.write_text(
    existing_readme.rstrip()
    + v3_docs,
    encoding="utf-8"
)

print("✅ README updated for V3")


In [ ]:

# 7. Static validation of the generated Streamlit app
import ast
import py_compile

# Python syntax checks.
py_compile.compile(
    str(SERVICE_ROOT / "streamlit_app.py"),
    doraise=True
)

py_compile.compile(
    str(SERVICE_ROOT / "run_demo.py"),
    doraise=True
)

source = (
    SERVICE_ROOT / "streamlit_app.py"
).read_text(encoding="utf-8")

required_ui_terms = [
    "st.text_input",
    "st.button",
    "ranking_score",
    "retrieval_score",
    "latency_ms",
    "st.image",
    "top_k",
    "candidate_k",
]

ui_checks = {
    term: term in source
    for term in required_ui_terms
}

print("STREAMLIT STATIC CHECKS")
print("=" * 70)

for k,v in ui_checks.items():
    print(("✅" if v else "❌"), k)

if not all(ui_checks.values()):
    raise AssertionError(
        "Missing required UI element(s)"
    )

print("\n✅ Streamlit app syntax and required UI checks passed")


In [ ]:

# 8. Re-run a real search through the packaged V2 engine
import sys
sys.path.insert(0, str(SERVICE_ROOT))

from amazon_search_service.search_engine import SearchEngine

engine = SearchEngine(
    artifact_dir=SERVICE_ROOT / "artifacts"
)

test_query = "best non drying lip stain"

test_result = engine.search(
    test_query,
    top_k=5,
    candidate_k=100
)

print("QUERY:", test_result["query"])
print(
    "Pipeline latency:",
    round(test_result["latency_ms"]["total"], 2),
    "ms"
)

display(
    pd.DataFrame(
        test_result["results"]
    )
)

if len(test_result["results"]) != 5:
    raise AssertionError(
        "Expected 5 demo results"
    )

if test_result["latency_ms"]["total"] >= 200:
    print(
        "⚠️ This single run exceeded 200 ms; "
        "use the V2 benchmark for acceptance."
    )
else:
    print("✅ Demo search is under 200 ms")

print("✅ SearchEngine remains functional after V3 packaging")


In [ ]:

# 9. Create V3 summary
from datetime import datetime

summary = {
    "week": 7,
    "version": "V3",
    "stage": "Streamlit Demo",
    "demo_features": {
        "natural_language_query": True,
        "top_k_control": True,
        "candidate_k_control": True,
        "ranked_product_title": True,
        "product_image_when_available": True,
        "price_rating_reviews": True,
        "ranking_score": True,
        "retrieval_score": True,
        "latency_breakdown": True,
        "latency_target_status": True,
        "direct_engine_mode": True,
        "fastapi_mode": True,
    },
    "catalog_products": 10000,
    "retrieval_candidates_default": 100,
    "ranker_features": 29,
    "v2_api_mean_ms": 92.56903489997512,
    "v2_api_p95_ms": 119.58991929988085,
    "v2_latency_target_pass": True,
    "generated_at_utc": datetime.utcnow().isoformat() + "Z",
}

evaluation_dir = SERVICE_ROOT / "evaluation"
evaluation_dir.mkdir(exist_ok=True)

with open(
    evaluation_dir / "week7_v3_summary.json",
    "w"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print(json.dumps(summary, indent=2))


In [ ]:

# 10. Package and download Week 7 V3
zip_base = "/content/amazon_week7_v3_streamlit_demo"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    SERVICE_ROOT
)

print("✅ Created:", zip_path)

print("\nKey files:")
for name in [
    "streamlit_app.py",
    "run_demo.py",
    "docker-compose.yml",
    "Dockerfile",
    "requirements.txt",
    "README.md",
    "amazon_search_service/main.py",
    "amazon_search_service/search_engine.py",
]:
    p = SERVICE_ROOT / name
    print(("✅" if p.exists() else "❌"), name)

files.download(zip_path)



# V3 completion

After this notebook succeeds, the package contains both the search backend and UI:

```text
User
 ↓
Streamlit UI
 ↓
FastAPI /search  (API mode)
 ↓
SBERT Query Embedding
 ↓
FAISS Top-100
 ↓
29 Runtime Features
 ↓
LambdaMART
 ↓
Top-K Ranked Products
```

The final V3 ZIP is:

`amazon_week7_v3_streamlit_demo.zip`

Next: **Week 7 V4 — Final Evaluation, basic tests, architecture diagram, and final Week 7 report package.**
